# [Scorio GPQA](https://huggingface.co/buckets/harimo/scorio-gpqa)

Scorio GPQA contains 1,152,000 attempts from four model configurations on a fixed sample of
3,600 superGPQA questions. The sample contains 72 fields with exactly 50 questions per
field. Every model has 80 attempts per question.

One Parquet file is one candidate pool. The Bucket includes top-20 token distributions;
project columns when those distributions are not needed.


## Check Storage Bucket dependencies

Direct Bucket loading requires `datasets>=5.0.0` and `huggingface_hub>=1.5.0`. Run this cell
before loading data. It installs only missing or outdated requirements. If it installs an
update, restart the kernel and run the notebook from the top.


In [1]:
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

from packaging.version import Version

minimum_versions = {
    "datasets": Version("5.0.0"),
    "huggingface_hub": Version("1.5.0"),
}

installed_versions = {}
requirements = []
for package, minimum in minimum_versions.items():
    try:
        installed = Version(version(package))
    except PackageNotFoundError:
        installed = None
    installed_versions[package] = installed
    if installed is None or installed < minimum:
        requirements.append(f"{package}>={minimum}")

if requirements:
    print("Installing:", ", ".join(requirements))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", *requirements]
    )
    print("Restart the kernel, then run the notebook from the top.")
else:
    for package, minimum in minimum_versions.items():
        print(f"{package}: {installed_versions[package]} (required >= {minimum})")


datasets: 5.0.1 (required >= 5.0.0)
huggingface_hub: 1.29.0 (required >= 1.5.0)


## Load one candidate pool with `datasets`


In [2]:
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from datasets import load_dataset
from IPython.display import display

from scorio import agg

bucket_name = "buckets/harimo/scorio-gpqa"
bucket_root = "hf://buckets/harimo/scorio-gpqa"
model_name = "gpt-oss-20b_low"
question_id = 0

data_files = {"super_gpqa": f"data/{model_name}/super_gpqa/q{question_id:04d}.parquet"}
stream = load_dataset(bucket_name, data_files=data_files, split="super_gpqa", streaming=True)
first = next(iter(stream))

print(first["task"], first["model_key"], first["full_data_id"], first["seed"])
print("field:", first["field"])
print("difficulty:", first["difficulty"])
print("columns:", len(first))


/tmp/scorio_uv_cache/archive-v0/qlDst6OSBXQ1PndVk4Wef/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


super_gpqa gpt-oss-20b_low 0 0
field: Aeronautical and Astronautical Science and Technology
difficulty: middle
columns: 49


## One complete 80-attempt pool


In [3]:
def pool_path(question_id, model=model_name):
    return f"{bucket_root}/data/{model}/super_gpqa/q{question_id:04d}.parquet"


pool = pq.read_table(pool_path(question_id)).to_pylist()
assert len(pool) == 80
assert [row["seed"] for row in pool] == list(range(80))

print("attempts:", len(pool))
print("discipline / field:", pool[0]["discipline"], "/", pool[0]["field"])
print("stage-local data_id:", pool[0]["data_id"])
print("global full_data_id:", pool[0]["full_data_id"])
print("uuid:", pool[0]["uuid"])
print("five most common answers:", Counter(row["evalscope_extracted_answer"] for row in pool).most_common(5))


attempts: 80
discipline / field: Engineering / Aeronautical and Astronautical Science and Technology
stage-local data_id: 0
global full_data_id: 0
uuid: 351a4176c19a4ad8b4aeb7f9ade9c7c4
five most common answers: [('D', 44), ('F', 35), ('A', 1)]


## List all fields

Fields occupy contiguous blocks of 50 `full_data_id` values. Reading the first question of
each block builds the 72-field catalog. This cell opens 72 Parquet files and requests three
metadata columns from each file.


In [4]:
def read_one_field_marker(question_id):
    return pq.read_table(
        pool_path(question_id),
        columns=["full_data_id", "discipline", "field"],
    ).slice(0, 1)


starts = list(range(0, 3600, 50))
with ThreadPoolExecutor(max_workers=2) as executor:
    markers = list(executor.map(read_one_field_marker, starts))

field_catalog = pa.concat_tables(markers).to_pandas()
field_catalog["start_full_data_id"] = starts
field_catalog = field_catalog[["discipline", "field", "start_full_data_id"]]

assert len(field_catalog) == 72
display(field_catalog)


,discipline,field,start_full_data_id
0,Engineering,Aeronautical and Astronautical Science and Tec...,0
1,Engineering,Agricultural Engineering,50
2,Agronomy,Animal Husbandry,100
3,Economics,Applied Economics,150
4,Agronomy,Aquaculture,200
...,...,...,...
67,Economics,Theoretical Economics,3350
68,Medicine,Traditional Chinese Medicine,3400
69,Engineering,Transportation Engineering,3450
70,Agronomy,Veterinary Medicine,3500


## Load one field or a list of fields

List the fields to load in `selected_fields`. The example reads 50 Parquet files for the first
field and requests only metadata and grades, leaving the top-20 token distributions remote.


In [5]:
def question_ids_for_fields(field_names):
    selected = field_catalog[field_catalog.field.isin(field_names)]
    missing = sorted(set(field_names) - set(selected.field))
    if missing:
        raise KeyError(f"unknown fields: {missing}")
    return [
        question_id
        for start in selected.start_full_data_id
        for question_id in range(int(start), int(start) + 50)
    ]


def read_selected_fields(field_names, columns, model=model_name):
    question_ids = question_ids_for_fields(field_names)

    def read_one(question_id):
        return pq.read_table(pool_path(question_id, model=model), columns=columns)

    with ThreadPoolExecutor(max_workers=2) as executor:
        tables = list(executor.map(read_one, question_ids))
    return pa.concat_tables(tables).to_pandas()


selected_fields = [field_catalog.iloc[0].field]  # add more field names to this list
field_rows = read_selected_fields(
    selected_fields,
    ["full_data_id", "seed", "difficulty", "evalscope_is_correct"],
)

print("fields:", selected_fields)
print("questions:", field_rows.full_data_id.nunique())
print("attempts:", len(field_rows))
display(field_rows.drop_duplicates("full_data_id").difficulty.value_counts())


fields: ['Aeronautical and Astronautical Science and Technology']
questions: 50
attempts: 4000


difficulty
hard      18
middle    17
easy      15
Name: count, dtype: int64

## Top-20 confidence signals for one attempt


In [6]:
attempt = pool[0]
topk = [[item["logprob"] for item in position]
        for position in attempt["tokens"]["completion_topk_logprobs_list"]]

signals = {
    "self_certainty": agg.self_certainty(topk),
    "deepconf": agg.deepconf_confidence(topk),
    "entropy": agg.token_entropy(topk),
    "varentropy": agg.varentropy(topk),
    "max_probability": agg.max_softmax_probability(topk),
    "logprob_margin": agg.logprob_margin(topk),
}
display(pd.Series(signals).round(4))


self_certainty      7.3558
deepconf           10.3565
entropy             0.6241
varentropy          0.6777
max_probability     0.7878
logprob_margin      5.0020
dtype: float64

Use `full_data_id`, `uuid`, `selection_hash`, or `(stage, data_id)` when joining records.
`data_id` alone is not unique across all 3,600 questions.
